In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch

import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
)

import optuna, optuna_dashboard
from optuna.trial import TrialState


from src.dataset import *
from src.lightning import *
from src.models import *
from src.params import *
from src.utils import *
from optuna_integration import PyTorchLightningPruningCallback


In [3]:
files_dir = PROCESSED_DIR / "cropped" / "files"
metadata = pd.read_csv(DATA_DIR / "train.csv")
dm = BrainDataModule(metadata=metadata, spec_dir= files_dir, batch_size= 32, num_workers= 8, verbose= False)

In [4]:
def objective(trial, datamodule):

    # def hyperparams to tune

    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    weight_decay = 1.2e-5
    dropout =   0.085
    mixup_alpha = 0.35
    n_blocks = 3
    kernel_size = trial.suggest_categorical("kernel_size",[5,7])
    hidden_dims = [trial.suggest_int(f"dim_{i}", 16, 256) for i in range(n_blocks * 2)]
    # prun o total hidden dims
    if sum(hidden_dims) > 632: #based on optimal candidate first run
        raise optuna.TrialPruned()

    # def model
    optuna_model = OptunaModel(n_channels= 4, n_classes= 6,
                               hidden_dims= hidden_dims,
                               kernel_size= kernel_size,
                               dropout= dropout)



    n_params = sum(p.numel() for p in optuna_model.parameters())
    print(f"Trial {trial.number} | kernel={kernel_size} | lr={learning_rate:.2e} | dims={hidden_dims} | n_params={n_params:,}")
    if n_params > 2_000_000:
        raise optuna.TrialPruned()



    # def lightning module
    lit_optuna = BrainLightning(model= optuna_model, n_classes= 6,
                                lr= learning_rate, mixup= True, mixup_alpha= mixup_alpha,
                               scheduler= True, t_max= 25, weight_decay= weight_decay, verbose= False)

    # def callbacks adapted to tuning (short)
    callbacks = [
    ModelCheckpoint(
        dirpath=f"checkpoints/optuna2/{trial.number}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        filename="{epoch:02d}-{val_loss:.3f}",
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min",
        min_delta=1e-3,
    ),
    PyTorchLightningPruningCallback(trial, monitor= "val_loss")
]



    trainer = pl.Trainer(
        max_epochs=25,
        accelerator="auto",
        devices="auto",
        callbacks=callbacks,
        logger= False,
        precision="bf16-mixed",
        enable_progress_bar= False, #silent
        log_every_n_steps=10,
    )

    trainer.fit(lit_optuna, datamodule= datamodule)

    val_loss_final = callbacks[0].best_model_score.item()
    return val_loss_final

In [ ]:
storage = optuna.storages.RDBStorage("sqlite:///optuna_brain_v2.db")
study = optuna.create_study(
    direction = "minimize",
    sampler= optuna.samplers.TPESampler(seed= 273),
    pruner= optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5, interval_steps=2 ),
    storage= storage,
    study_name= "brain_optuna_v2",
    load_if_exists= True
)

study.optimize(lambda trial: objective(trial, dm), n_trials= 300, show_progress_bar= True)

trials = study.trials
n_complete = len([t for t in trials if t.state == TrialState.COMPLETE])
n_pruned   = len([t for t in trials if t.state == TrialState.PRUNED])
n_failed   = len([t for t in trials if t.state == TrialState.FAIL])

print(f"Complétés : {n_complete}")
print(f"Pruned    : {n_pruned}")
print(f"Failed    : {n_failed}")

print(f"Best score : {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

[I 2026-04-12 17:08:52,244] A new study created in RDB with name: brain_optuna_v2


  0%|          | 0/300 [00:00<?, ?it/s]

[I 2026-04-12 17:08:52,266] Trial 0 pruned. 
[I 2026-04-12 17:08:52,278] Trial 1 pruned. 
[I 2026-04-12 17:08:52,290] Trial 2 pruned. 
Trial 3 | kernel=7 | lr=9.03e-03 | dims=[35, 174, 152, 139, 28, 67] | n_params=2,925,058
[I 2026-04-12 17:08:52,306] Trial 3 pruned. 
Trial 4 | kernel=7 | lr=3.31e-03 | dims=[25, 163, 168, 46, 153, 76] | n_params=2,846,277
[I 2026-04-12 17:08:52,324] Trial 4 pruned. 
[I 2026-04-12 17:08:52,336] Trial 5 pruned. 
[I 2026-04-12 17:08:52,347] Trial 6 pruned. 
[I 2026-04-12 17:08:52,359] Trial 7 pruned. 
[I 2026-04-12 17:08:52,369] Trial 8 pruned. 
[I 2026-04-12 17:08:52,380] Trial 9 pruned. 
[I 2026-04-12 17:08:52,394] Trial 10 pruned. 
[I 2026-04-12 17:08:52,409] Trial 11 pruned. 
[I 2026-04-12 17:08:52,425] Trial 12 pruned. 
[I 2026-04-12 17:08:52,440] Trial 13 pruned. 
[I 2026-04-12 17:08:52,455] Trial 14 pruned. 
[I 2026-04-12 17:08:52,473] Trial 15 pruned. 
[I 2026-04-12 17:08:52,490] Trial 16 pruned. 
[I 2026-04-12 17:08:52,507] Trial 17 pruned. 
[I 2

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[I 2026-04-12 17:08:52,677] Trial 27 pruned. 
[I 2026-04-12 17:08:52,695] Trial 28 pruned. 
[I 2026-04-12 17:08:52,711] Trial 29 pruned. 
Trial 30 | kernel=7 | lr=7.04e-03 | dims=[73, 79, 206, 142, 55, 41] | n_params=3,025,120
[I 2026-04-12 17:08:52,735] Trial 30 pruned. 
[I 2026-04-12 17:08:52,750] Trial 31 pruned. 
[I 2026-04-12 17:08:52,766] Trial 32 pruned. 
[I 2026-04-12 17:08:52,783] Trial 33 pruned. 
[I 2026-04-12 17:08:52,801] Trial 34 pruned. 
Trial 35 | kernel=5 | lr=2.24e-03 | dims=[27, 108, 35, 71, 98, 212] | n_params=940,699
[setup] Loading splits from cache: /Users/pablorougerie/code/Projets/brain_waves/data/cache/5_at_seed_273_min_vote_1.json
[setup] Cache loaded — 5 folds found


...

Trainable params: 672 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 672 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 48                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/Users/pablorougerie/code/Projets/brain_waves/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
python3(83567) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(83568) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(83569) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(83570) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(83572) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(83744) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(86289) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(86293) MallocStackLogging: can't turn off malloc stack log

Epoch 000 | val_loss:   1.3779


python3(86335) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(86349) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(86351) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(86352) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(86353) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(86355) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(86415) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3(87040) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Epoch 000 | val_loss:   1.1931
Epoch 001 | val_loss:   0.9533
Epoch 002 | val_loss:   0.9651
Epoch 003 | val_loss:   0.8359
Epoch 004 | val_loss:   0.8040
Epoch 005 | val_loss:   0.8432
Epoch 006 | val_loss:   0.7689
Epoch 007 | val_loss:   0.7827
Epoch 008 | val_loss:   0.7708
Epoch 009 | val_loss:   0.7825


In [7]:
storage = optuna.storages.RDBStorage("sqlite:///optuna_brain_v2.db")
study = optuna.load_study(study_name="brain_optuna_v2", storage=storage)

best = study.best_trial
print(best.value)   # 0.6764
print(best.params)  # learning_rate, kernel_size, dim_0..dim_5
print(best.number)  # → 59


0.676441490650177
{'learning_rate': 0.0028273119832751066, 'kernel_size': 5, 'dim_0': 117, 'dim_1': 52, 'dim_2': 60, 'dim_3': 64, 'dim_4': 103, 'dim_5': 226}
59


In [8]:
import plotly

In [9]:
storage = optuna.storages.RDBStorage("sqlite:///optuna_brain_v2.db")
study = optuna.load_study(study_name="brain_optuna_v2", storage=storage)

In [10]:
optuna.visualization.plot_optimization_history(study)

In [11]:
optuna.visualization.plot_param_importances(study)


In [12]:
optuna.visualization.plot_parallel_coordinate(study)